# BigQuery run — crypto_ethereum public dataset

Runs one honest-scale query over Google's public Ethereum dataset
(`bigquery-public-data.crypto_ethereum`) for our four token contracts.

**Cost discipline (read before running):**
- Cell 2 performs a **dry run** — it returns bytes scanned without
  executing or billing. Check the number before running Cell 3.
- On-demand BigQuery pricing is ~$6.25/TiB scanned (verify current
  pricing). If the dry run reports more than a few GB, narrow the
  `block_timestamp` range in the query before executing.
- Auth: `gcloud auth application-default login`, or set
  `GOOGLE_APPLICATION_CREDENTIALS` to a service-account JSON path.
  The free 1 TiB/month sandbox tier usually covers this query.

**Honesty rule:** record the real bytes scanned and row counts in the
write-up. Do not call this 'petabyte-scale' — state what it actually is.

In [ ]:
# Install once: pip install google-cloud-bigquery pandas db-dtypes
from google.cloud import bigquery

client = bigquery.Client()
print("Project:", client.project)

In [ ]:
QUERY = """
SELECT
    token_address,
    DATE(block_timestamp) AS day,
    COUNT(*) AS transfers,
    COUNT(DISTINCT from_address) AS senders,
    COUNT(DISTINCT to_address) AS receivers
FROM `bigquery-public-data.crypto_ethereum.token_transfers`
WHERE token_address IN (
    '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48',  -- USDC
    '0x7712c34205737192402172409a8f7ccef8aa2aec',  -- BUIDL
    '0x1b19c19393e2d034d8ff31ff34c81252fcbbee92',  -- OUSG
    '0x96f6ef951840721adbf46ac996b59e0235cb985c'   -- USDY
)
  AND block_timestamp >= TIMESTAMP('2025-01-01')
GROUP BY token_address, day
ORDER BY token_address, day
"""

# BigQuery stores addresses lowercase — the values above are the
# checksummed addresses from docs/scope.md lowercased.

job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry = client.query(QUERY, job_config=job_config)
print(f"Dry run: {dry.total_bytes_processed / 1e9:.2f} GB scanned "
      f"(~${dry.total_bytes_processed / 1e12 * 6.25:.4f} at $6.25/TiB)")

If the dry-run size is acceptable, run the real query below and record
the actual bytes processed (`job.total_bytes_processed`) in the write-up.

In [ ]:
job = client.query(QUERY)
df = job.result().to_dataframe()
print(f"Rows: {len(df)}")
print(f"Bytes processed: {job.total_bytes_processed:,}")
df.head()